In [ ]:
#!pip install tabpfn

In [ ]:
import os

# Paths shown reflect the default Jupyter Docker Stacks user directory (/home/jovyan).
code_path = '/home/jovyan/code/'

# source utility functions 
file_path = os.path.join(code_path, 'utility_functions_implementing_tabpfn_generators_iclr.py')
with open(os.path.expanduser(file_path)) as file:
    exec(file.read())

# source additional utility functions 
file_path = os.path.join(code_path, 'additional_utility_functions_for_tabpfn_generators_iclr.py')
with open(os.path.expanduser(file_path)) as file:
    exec(file.read())

In [ ]:
import os                              # filesystem paths and directory creation
import pandas as pd                    # Pandas
import openml                          # OpenML API client
from tqdm import tqdm                  # progress bar
from sklearn.model_selection import train_test_split  # random data splitting
import re
import pyarrow.feather as feather
import time

In [ ]:
# Select a subset of the OpenMLCC18 datasets (<= 2000 columns, <= 100 columns, <= 10 classes)

# Load the OpenML study suite
suite = openml.study.get_suite("OpenML-CC18")   # alias for study id 99
task_ids = suite.tasks  # list of task IDs

# Filtering thresholds
n_max = 2000         # max rows
n_cols_max = 100     # max columns
n_class_max = 10     # max number of classes (levels) among categorical columns

tasks_to_keep = []

for tsk in tqdm(task_ids, desc="Dataset"):
    # Get task and underlying dataset
    task = openml.tasks.get_task(tsk)
    dataset = task.get_dataset()

    # Get a SINGLE DataFrame with ALL columns (features + target)
    # y will be None because we don't split targets out
    X, y, categorical_mask, attr_names = dataset.get_data(
        target=None, dataset_format="dataframe"
    )

    # Basic shape
    n_rows, n_cols = X.shape

    # Identify non-numeric columns (category, string, object, bool, etc.)
    non_num_cols = X.select_dtypes(exclude="number").columns
    n_non_num = len(non_num_cols)
    n_num = n_cols - n_non_num

    # Count levels (unique values) per non-numeric column; ignore NaN in the count
    if n_non_num > 0:
        levels_per_cat = X[non_num_cols].nunique(dropna=True)
        max_classes = int(levels_per_cat.max())
    else:
        max_classes = 0  # no categorical columns

    # Keep datasets that satisfy:
    # - size constraints
    # - more numeric than non-numeric columns
    # - maximum number of categorical levels <= n_class_max
    if (
        (n_rows <= n_max)
        and (n_cols <= n_cols_max)
        and (n_num > n_non_num)
        and (max_classes <= n_class_max)
    ):
        tasks_to_keep.append(tsk)


len(tasks_to_keep)

In [ ]:
# Select a larger subset of the OpenMLCC18 datasets (<= 10000 columns, <= 500 columns, <= 10 classes)

# Load the OpenML study suite
suite = openml.study.get_suite("OpenML-CC18")   # alias for study id 99
task_ids = suite.tasks  # list of task IDs

# Filtering thresholds
n_max = 10000         # max rows
n_cols_max = 500     # max columns
n_class_max = 10     # max number of classes (levels) among categorical columns

tasks_to_keep_2 = []

for tsk in tqdm(task_ids, desc="Dataset"):
    # Get task and underlying dataset
    task = openml.tasks.get_task(tsk)
    dataset = task.get_dataset()

    # Get a SINGLE DataFrame with ALL columns (features + target)
    # y will be None because we don't split targets out
    X, y, categorical_mask, attr_names = dataset.get_data(
        target=None, dataset_format="dataframe"
    )

    # Basic shape
    n_rows, n_cols = X.shape

    # Identify non-numeric columns (category, string, object, bool, etc.)
    non_num_cols = X.select_dtypes(exclude="number").columns
    n_non_num = len(non_num_cols)
    n_num = n_cols - n_non_num

    # Count levels (unique values) per non-numeric column; ignore NaN in the count
    if n_non_num > 0:
        levels_per_cat = X[non_num_cols].nunique(dropna=True)
        max_classes = int(levels_per_cat.max())
    else:
        max_classes = 0  # no categorical columns

    # Keep datasets that satisfy:
    # - size constraints
    # - more numeric than non-numeric columns
    # - maximum number of categorical levels <= n_class_max
    if (
        (n_rows <= n_max)
        and (n_cols <= n_cols_max)
        and (n_num > n_non_num)
        and (max_classes <= n_class_max)
    ):
        tasks_to_keep_2.append(tsk)


len(tasks_to_keep_2)

In [ ]:
# Get the additional tasks (additional ones that where not analysed before)

def setdiff_ordered(x, y):
    yset = set(y)
    seen = set()
    out = []
    for v in x:
        if v not in yset and v not in seen:
            seen.add(v)
            out.append(v)
    return out


additional_tasks_to_keep = setdiff_ordered(tasks_to_keep_2, tasks_to_keep)

len(additional_tasks_to_keep)

In [ ]:
summary = summarize_openml_tasks(additional_tasks_to_keep)
summary

In [ ]:
split_seeds = list(range(1, 11)) # 10 splits

In [ ]:
# Generate one big file with all the original and holdout data splits (for additional tasks)

t0 = time.perf_counter()
_, splits_long = build_all_splits_tasks(additional_tasks_to_keep, split_seeds)
elapsed = time.perf_counter() - t0

# Save the file 
feather.write_feather(splits_long, "/home/jovyan/selected_OpenMLCC18/outputs/openml_cc18_orig_hold_data_splits_additional.feather")

print(f"elapsed running time: {elapsed/60:.2f} minutes " f"(~{elapsed/3600:.2f} hours)")

In [ ]:
# Generate one big file with all MIAV synthetic datasets (for additional tasks)

t0 = time.perf_counter()
syn_long, _ = build_all_synthetics_miav_tasks(
    tasks_to_keep=additional_tasks_to_keep,
    split_seeds=split_seeds
)
elapsed = time.perf_counter() - t0

# Save the file
feather.write_feather(syn_long, "/home/jovyan/selected_OpenMLCC18/outputs/openml_cc18_syn_miav_additional.feather")

print(f"elapsed running time: {elapsed/60:.2f} minutes " f"(~{elapsed/3600:.2f} hours)")

In [ ]:
# Generate one big file with all noisy MIAV synthetic datasets (percent = 0.05) (for additional tasks)

t0 = time.perf_counter()
syn_long, _ = build_all_synthetics_noisy_miav_tasks(
    tasks_to_keep=additional_tasks_to_keep,
    split_seeds=split_seeds,
    percent=0.05
)
elapsed = time.perf_counter() - t0

# Save the file
feather.write_feather(syn_long, "/home/jovyan/selected_OpenMLCC18/outputs/openml_cc18_syn_noisy_miav_0.05_additional.feather")

print(f"elapsed running time: {elapsed/60:.2f} minutes " f"(~{elapsed/3600:.2f} hours)")

In [ ]:
# Generate one big file with all noisy MIAV synthetic datasets (percent = 0.1) (for additional tasks)

t0 = time.perf_counter()
syn_long, _ = build_all_synthetics_noisy_miav_tasks(
    tasks_to_keep=additional_tasks_to_keep,
    split_seeds=split_seeds,
    percent=0.1
)
elapsed = time.perf_counter() - t0

# Save the file
feather.write_feather(syn_long, "/home/jovyan/selected_OpenMLCC18/outputs/openml_cc18_syn_noisy_miav_0.1_additional.feather")

print(f"elapsed running time: {elapsed/60:.2f} minutes " f"(~{elapsed/3600:.2f} hours)")

In [ ]:
# Generate one big file with all noisy MIAV synthetic datasets (percent = 0.15) (for additional tasks)

t0 = time.perf_counter()
syn_long, _ = build_all_synthetics_noisy_miav_tasks(
    tasks_to_keep=additional_tasks_to_keep,
    split_seeds=split_seeds,
    percent=0.15
)
elapsed = time.perf_counter() - t0

# Save the file
feather.write_feather(syn_long, "/home/jovyan/selected_OpenMLCC18/outputs/openml_cc18_syn_noisy_miav_0.15_additional.feather")

print(f"elapsed running time: {elapsed/60:.2f} minutes " f"(~{elapsed/3600:.2f} hours)")

In [ ]:
# Generate one big file with all noisy MIAV synthetic datasets (percent = 0.2) (for additional tasks)

t0 = time.perf_counter()
syn_long, _ = build_all_synthetics_noisy_miav_tasks(
    tasks_to_keep=additional_tasks_to_keep,
    split_seeds=split_seeds,
    percent=0.2
)
elapsed = time.perf_counter() - t0

# Save the file
feather.write_feather(syn_long, "/home/jovyan/selected_OpenMLCC18/outputs/openml_cc18_syn_noisy_miav_0.2_additional.feather")

print(f"elapsed running time: {elapsed/60:.2f} minutes " f"(~{elapsed/3600:.2f} hours)")

In [ ]:
# Generate one big file with all noisy MIAV synthetic datasets (percent = 0.25) (for additional tasks)

t0 = time.perf_counter()
syn_long, _ = build_all_synthetics_noisy_miav_tasks(
    tasks_to_keep=additional_tasks_to_keep,
    split_seeds=split_seeds,
    percent=0.25
)
elapsed = time.perf_counter() - t0

# Save the file
feather.write_feather(syn_long, "/home/jovyan/selected_OpenMLCC18/outputs/openml_cc18_syn_noisy_miav_0.25_additional.feather")

print(f"elapsed running time: {elapsed/60:.2f} minutes " f"(~{elapsed/3600:.2f} hours)")

In [ ]:
# Generate one big file with all noisy MIAV synthetic datasets (percent = 0.3) (for additional tasks)

t0 = time.perf_counter()
syn_long, _ = build_all_synthetics_noisy_miav_tasks(
    tasks_to_keep=additional_tasks_to_keep,
    split_seeds=split_seeds,
    percent=0.3
)
elapsed = time.perf_counter() - t0

# Save the file
feather.write_feather(syn_long, "/home/jovyan/selected_OpenMLCC18/outputs/openml_cc18_syn_noisy_miav_0.3_additional.feather")

print(f"elapsed running time: {elapsed/60:.2f} minutes " f"(~{elapsed/3600:.2f} hours)")

In [ ]:
# Generate one big file with all JF synthetic datasets (for additional tasks)

t0 = time.perf_counter()
syn_long, _ = build_all_synthetics_jf_tasks(
    tasks_to_keep=additional_tasks_to_keep,
    split_seeds=split_seeds
)
elapsed = time.perf_counter() - t0

# Save the file
feather.write_feather(syn_long, "/home/jovyan/selected_OpenMLCC18/outputs/openml_cc18_syn_jf_additional.feather")

print(f"elapsed running time: {elapsed/60:.2f} minutes " f"(~{elapsed/3600:.2f} hours)")

In [ ]:
# Generate one big file with all FC synthetic datasets (for additional tasks)

t0 = time.perf_counter()
syn_long, _ = build_all_synthetics_fc_tasks(
    tasks_to_keep=additional_tasks_to_keep,
    split_seeds=split_seeds
)
elapsed = time.perf_counter() - t0

# Save the file
feather.write_feather(syn_long, "/home/jovyan/selected_OpenMLCC18/outputs/openml_cc18_syn_fc_additional.feather")

print(f"elapsed running time: {elapsed/60:.2f} minutes " f"(~{elapsed/3600:.2f} hours)")